In [1]:
import pandas as pd
import numpy as np
from scipy.stats import poisson

In [2]:
team_stats = pd.read_csv('../data/team_stats.csv')
print(len(team_stats))
print(team_stats.columns.to_list())

48
['team', 'total_games', 'avg_goal_scored', 'avg_goal_conceded', 'win_rate', 'draw_rate', 'home_goals_scored', 'home_goals_conceded', 'home_win_rate', 'away_goals_scored', 'away_goals_conceded', 'away_win_rate', 'fifa_rank', 'fifa_points']


Things to be clear before starting further steps.
The psychology behind home advantage: Why home team performs better than away teams on their own ground
This is not a coincidence but its the result of a blend of psychological, environment and tactical factors.
Few of the factors can be:
- Familiarity with the env: pitch conditions, stadium layout, atmosphere
- No travel fatigue and routine disruption as away team has to travel and their routine might get disturbed due to unfamiliar env
- Crowd Support and Psychological boost

Next thing:
- Types of statistical models used in football prediction algorithms
a. Poisson regression (which we are going to use)
b. Gradient boosting
c. Neural Networks
- Poisson Distribution
Poisson distribution is mathematically well-suited to football because it models the probability of discrete rare events.
Poisson distribution calculates the prob of a specific number of events occurring within a fixed period, given a know average rate for that event(lambda).
      - Here, event if the goal, the fixed period is a 90-minute match (it works for interval, not sub interval).
     - If a team scores an avg of 1.5 goals per match the poisson dist calculates the exact prob of them scoring 0, 1, 2, and more goals in any given fixture based on that avg rate.
     - The Poisson dist stat model is used in football because of the assumption that goals behave like poisson process, each goal is approx independent of the previous one in terms of timing.
- Gradient boosting models handle complex non-linear relationships between structured data inputs without requiring those relationships to be specified manually. 

- Neural networks generalize well across large multi-league datasets when trained on sufficient volume of event-level data.

- But for the better performance, ensemble model is suggested. An ensemble model takes the output of the multiple individual models and combines them, through a weighted avg, to produce a final prob.


#### Building the score predictor function

In [5]:
def predict_match(home_team, away_team, team_stats, max_goals = 6):
    """
    Predicting the scoreline between two teams.
    Returns the expected goals and win probabilities
    """
    home = team_stats[team_stats['team']==home_team].iloc[0]
    away = team_stats[team_stats['team']==away_team].iloc[0]

    #calculating expected goals
    #Home expected goals
    # Dividing by 0.9 which is < 1 to slightly increase home scoring
    #because home teams usually performs better (home advantage assumption)
    home_expected_goals = (
        home['home_goals_scored'] * away['away_goals_conceded'] / 0.9
    )

    # /1.1 to slightly reduce scoring of away team
    away_expected_goals = (
        away['away_goals_scored'] * home['home_goals_conceded'] / 1.1
    )

    #Clipping to reasonable range of expected goals
    #np.clip(value, min, max)
    home_expected_goals = np.clip(home_expected_goals, 0.3, 4.0)
    away_expected_goals = np.clip(away_expected_goals, 0.3, 4.0)

    # Using poisson to build prob matrix
    home_prob = [poisson.pmf(i, home_expected_goals) for i in range(max_goals+1)]
    away_prob = [poisson.pmf(i, away_expected_goals) for i in range(max_goals+1)]

    #Building the score prob matrix
    #np.outer(a, b, out=None(optional)) computes the outer product of two vectors
    #a:[array_like], first input vector, input is flattened if not in 1D
    #b:[array_like], second input vector, input is flattened if not in 1D
    #out:[ndarray], it is a location where the result is stored
    score_matrix = np.outer(home_prob, away_prob)

    #Win probabilities
    home_win_prob = np.tril(score_matrix, -1).sum() #returns lower triangle of matrix
    draw_prob = np.trace(score_matrix) # diagonal elements
    away_win_prob = np.triu(score_matrix, 1).sum() #returns the upper triangle of matrix

    max_idx = np.unravel_index(score_matrix.argmax(), score_matrix.shape)
    predicted_home_goals = max_idx[0]
    predicted_away_goals = max_idx[1]


    if home_win_prob > away_win_prob and home_win_prob > draw_prob:
        winner = "home"
    elif away_win_prob > home_win_prob and away_win_prob > draw_prob:
        winner = "away"
    else:
        winner = "draw"

    return{
        'home_team':home_team,
        'away_team':away_team,
        'predicted_home_goals':predicted_home_goals,
        'predicted_away_goals':predicted_away_goals,
        'home_expected_goals':round(home_expected_goals, 2),
        'away_expected_goals':round(away_expected_goals, 2),
        'home_win_prob':round(home_win_prob * 100, 1),
        'draw_prob':round(draw_prob * 100, 1),
        'away_win_prob':round(away_win_prob * 100, 1),
        'predicted_winner':winner
    }

#Testing
result = predict_match("Brazil", "Germany", team_stats)
print(result)


{'home_team': 'Brazil', 'away_team': 'Germany', 'predicted_home_goals': np.int64(3), 'predicted_away_goals': np.int64(1), 'home_expected_goals': np.float64(4.0), 'away_expected_goals': np.float64(1.2), 'home_win_prob': np.float64(73.9), 'draw_prob': np.float64(8.7), 'away_win_prob': np.float64(6.3), 'predicted_winner': 'home'}


In [11]:
# Testing few matchups

matchups =[
    ('Argentina', 'France'),
    ('Brazil', 'Spain'),
    ('England', 'Germany'),
    ('Morocco', 'Portugal'),
]


for home, away in matchups:
    r = predict_match(home, away, team_stats)
    score = f"{r['predicted_home_goals']}-{r['predicted_away_goals']}"
    print(f"{home:<20} {away:<20} {score:<20} {r['home_win_prob']:<10} {r['draw_prob']:<10} {r['away_win_prob']:<10} {r['predicted_winner']}")

Argentina            France               2-0                  72.0       17.5       9.7        home
Brazil               Spain                2-1                  63.2       19.3       16.7       home
England              Germany              3-1                  69.6       12.9       11.9       home
Morocco              Portugal             1-0                  53.3       28.3       18.3       home
